# Lesson 8: エンベロープ（ADSR）

**コンパニオンノートブック** — 詳しい解説は本文 Lesson 8 を参照してください。

## セットアップ

In [ ]:
# --- 最初に1回だけ実行 ---
import sys
try:
    import google.colab
    !pip install -q japanize-matplotlib
    !git clone -q https://github.com/ggszk/simple-sound-programming.git
    sys.path.append('/content/simple-sound-programming')
except ImportError:
    sys.path.append('..')

from audio_lib.notebook import setup_environment
setup_environment()

## このレッスンで学ぶこと

- エンベロープの概念を理解する
- ADSR（Attack, Decay, Sustain, Release）の4つの段階を学ぶ
- Python でエンベロープを音に適用し、表情をつける体験をする
- エンベロープを自分で実装して仕組みを理解する
- エンベロープの違いが音の印象にどう影響するかを体験する


### 突然始まる音の問題

In [ ]:
import numpy as np
from IPython.display import display
from audio_lib import sine_wave, AudioSignal
from audio_lib.notebook import play_sound

# 0.5秒のサイン波（エンベロープなし）
sig = sine_wave(440, 0.5)
display(play_sound(sig, "サイン波（エンベロープなし）"))

## 8.3 エンベロープを音に適用する

In [ ]:
from audio_lib import adsr, sine_wave, AudioSignal
from audio_lib.notebook import play_sound
from IPython.display import display

# ADSR エンベロープを生成
# duration は波形と同じ長さにする（内部で gate_time = duration - release として計算される）
dur = 1.5
env = adsr(dur, attack=0.1, decay=0.2, sustain=0.6, release=0.3)

# サイン波を生成（エンベロープと同じ長さ）
wave = sine_wave(440, dur)

# 波形 × エンベロープ（要素ごとの掛け算）
shaped = AudioSignal(wave.data * env.data, 44100)

In [ ]:
display(play_sound(wave, "エンベロープなし"))
display(play_sound(shaped, "エンベロープあり"))

### さまざまな楽器のエンベロープ

In [ ]:
from audio_lib import sawtooth_wave

dur = 2.0
freq = 262  # C4

# ピアノ: 瞬間的な立ち上がり、長い減衰
env_piano = adsr(dur, attack=0.01, decay=0.5, sustain=0.2, release=0.3)
wave_piano = sawtooth_wave(freq, dur)
piano = AudioSignal(wave_piano.data * env_piano.data, 44100)
display(play_sound(piano, "ピアノ風（A=0.01, D=0.5, S=0.2, R=0.3）"))

# バイオリン: ゆっくり立ち上がり、高いサステイン
env_violin = adsr(dur, attack=0.3, decay=0.1, sustain=0.8, release=0.4)
wave_violin = sawtooth_wave(freq, dur)
violin = AudioSignal(wave_violin.data * env_violin.data, 44100)
display(play_sound(violin, "バイオリン風（A=0.3, D=0.1, S=0.8, R=0.4）"))

# オルガン: 素早い立ち上がり、サステインが最大
env_organ = adsr(dur, attack=0.01, decay=0.0, sustain=1.0, release=0.05)
wave_organ = sawtooth_wave(freq, dur)
organ = AudioSignal(wave_organ.data * env_organ.data, 44100)
display(play_sound(organ, "オルガン風（A=0.01, D=0, S=1.0, R=0.05）"))

# パーカッション: 瞬間的な立ち上がりと急速な減衰、サステインなし
env_perc = adsr(dur, attack=0.005, decay=0.15, sustain=0.0, release=0.1)
wave_perc = sine_wave(freq, dur)
perc = AudioSignal(wave_perc.data * env_perc.data, 44100)
display(play_sound(perc, "パーカッション風（A=0.005, D=0.15, S=0, R=0.1）"))

### エンベロープの比較

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 6))

envelopes = [
    ("ピアノ風", env_piano),
    ("バイオリン風", env_violin),
    ("オルガン風", env_organ),
    ("パーカッション風", env_perc),
]

for ax, (name, env) in zip(axes.flat, envelopes):
    t = np.arange(len(env.data)) / 44100
    ax.plot(t, env.data, linewidth=2)
    ax.set_title(name)
    ax.set_ylim(-0.05, 1.1)
    ax.set_xlabel("時間 (秒)")
    ax.set_ylabel("レベル")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8.4 波形とエンベロープの関係を可視化する

In [ ]:
dur = 1.5
wave = sine_wave(440, dur)
env = adsr(dur, attack=0.1, decay=0.2, sustain=0.6, release=0.3)
shaped = AudioSignal(wave.data * env.data, 44100)

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

t = np.arange(len(wave.data)) / 44100

# 元の波形
axes[0].plot(t, wave.data, alpha=0.7)
axes[0].set_ylabel("振幅")
axes[0].set_title("元の波形（サイン波）")
axes[0].set_ylim(-1.2, 1.2)

# エンベロープ
axes[1].plot(t, env.data, color='orange', linewidth=2)
axes[1].set_ylabel("レベル")
axes[1].set_title("ADSR エンベロープ")
axes[1].set_ylim(-0.1, 1.2)

# エンベロープ適用後
axes[2].plot(t, shaped.data, alpha=0.7, color='green')
axes[2].plot(t, env.data, color='orange', linewidth=1, linestyle='--', alpha=0.5)
axes[2].plot(t, -env.data, color='orange', linewidth=1, linestyle='--', alpha=0.5)
axes[2].set_ylabel("振幅")
axes[2].set_xlabel("時間 (秒)")
axes[2].set_title("エンベロープ適用後")
axes[2].set_ylim(-1.2, 1.2)

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8.5 エンベロープでメロディを鳴らす

In [ ]:
from audio_lib import note_to_frequency

c_major = [60, 62, 64, 65, 67, 69, 71, 72]
dur_note = 0.5  # 各音の長さ

parts = []
for midi_num in c_major:
    freq = note_to_frequency(midi_num)
    wave = sine_wave(freq, dur_note)
    env = adsr(dur_note, attack=0.02, decay=0.1, sustain=0.6, release=0.1)
    shaped = AudioSignal(wave.data * env.data, 44100)
    parts.append(shaped.data)

scale_signal = AudioSignal(np.concatenate(parts), 44100)
display(play_sound(scale_signal, "C メジャースケール（ADSR あり）"))

### 波形を変えてみる

In [ ]:
parts_saw = []
for midi_num in c_major:
    freq = note_to_frequency(midi_num)
    wave = sawtooth_wave(freq, dur_note)
    env = adsr(dur_note, attack=0.01, decay=0.15, sustain=0.4, release=0.1)
    shaped = AudioSignal(wave.data * env.data, 44100)
    parts_saw.append(shaped.data)

scale_saw = AudioSignal(np.concatenate(parts_saw), 44100)
display(play_sound(scale_saw, "C メジャースケール（ノコギリ波 + ADSR）"))

## 8.6 エンベロープの仕組みを理解する — 自分で実装してみよう

In [ ]:
def make_adsr_linear(duration, attack, decay, sustain, release,
                     sample_rate=44100):
    """直線的な ADSR エンベロープを生成"""
    num_samples = int(sample_rate * duration)
    gate_time = duration - release  # ゲートが開いている時間
    envelope = np.zeros(num_samples)

    for i in range(num_samples):
        t = i / sample_rate

        if t < attack:
            # Attack: 0 → 1（直線的に上昇）
            envelope[i] = t / attack
        elif t < attack + decay:
            # Decay: 1 → sustain（直線的に下降）
            envelope[i] = 1.0 - (1.0 - sustain) * (t - attack) / decay
        elif t < gate_time:
            # Sustain: sustain レベルを維持
            envelope[i] = sustain
        else:
            # Release: sustain → 0（直線的に下降）
            elapsed = t - gate_time
            if elapsed < release:
                envelope[i] = sustain * (1.0 - elapsed / release)

    return AudioSignal(envelope, sample_rate)

### 可視化して確認する

In [ ]:
env = make_adsr_linear(1.5, attack=0.1, decay=0.2, sustain=0.6, release=0.3)

plt.figure(figsize=(10, 4))
t = np.arange(len(env.data)) / 44100
plt.plot(t, env.data)
plt.xlabel("時間 (秒)")
plt.ylabel("振幅")
plt.title("ADSR エンベロープ（リニア・自作）")
plt.grid(True, alpha=0.3)

# 各段階にラベルをつける
plt.annotate("Attack", xy=(0.05, 0.5), fontsize=12, ha='center')
plt.annotate("Decay", xy=(0.2, 0.85), fontsize=12, ha='center')
plt.annotate("Sustain", xy=(0.7, 0.65), fontsize=12, ha='center')
plt.annotate("Release", xy=(1.35, 0.3), fontsize=12, ha='center')

plt.tight_layout()
plt.show()

### 音にして聞いてみる

In [ ]:
wave = sine_wave(440, 1.5)
env_lin = make_adsr_linear(1.5, attack=0.1, decay=0.2, sustain=0.6, release=0.3)
shaped_lin = AudioSignal(wave.data * env_lin.data, 44100)
display(play_sound(shaped_lin, "自作リニアエンベロープ"))

## 8.7 リニア vs 指数カーブ

In [ ]:
dur = 1.5
env_linear = make_adsr_linear(dur, attack=0.2, decay=0.3, sustain=0.5, release=0.3)
env_exp = adsr(dur, attack=0.2, decay=0.3, sustain=0.5, release=0.3)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

t_lin = np.arange(len(env_linear.data)) / 44100
t_exp = np.arange(len(env_exp.data)) / 44100

axes[0].plot(t_lin, env_linear.data, linewidth=2)
axes[0].set_title("リニア（直線的・自作）")
axes[0].set_xlabel("時間 (秒)")
axes[0].set_ylabel("レベル")
axes[0].set_ylim(-0.05, 1.1)
axes[0].grid(True, alpha=0.3)

axes[1].plot(t_exp, env_exp.data, linewidth=2, color='orange')
axes[1].set_title("指数カーブ（audio_lib）")
axes[1].set_xlabel("時間 (秒)")
axes[1].set_ylabel("レベル")
axes[1].set_ylim(-0.05, 1.1)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 聞き比べてみよう

In [ ]:
wave = sawtooth_wave(440, dur)
shaped_lin = AudioSignal(wave.data * env_linear.data, 44100)
shaped_exp = AudioSignal(wave.data * env_exp.data, 44100)

display(play_sound(shaped_lin, "リニアエンベロープ"))
display(play_sound(shaped_exp, "指数カーブエンベロープ"))